# 00 · Basics — RAG Retriever Concepts

**Start here.** This notebook covers the foundations every later method assumes:

1. Install + API key  
2. Load PDF → enrich metadata → chunk  
3. Embeddings + Chroma vector store  
4. Dense similarity search  
5. `search_type` knobs: similarity, MMR, score threshold  
6. Similarity metrics (cosine / Euclidean / dot)  
7. Metadata pre-filtering vs post-filtering  

**Then open a method notebook** from `method-notebooks/` (HyDE, Multi-Query, RRF, …).

Companion analogies: [Retriever Analogy Handbook](../retriever-analogy-handbook.html)


### Learning: Install Dependencies

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** First, we need to install the necessary libraries for this notebook, including `langchain_community`, `langchain_text_splitters`, `langchain_openai`, `langchain_chroma`, and `pypdf`.

**Watch for:** Run once; restart runtime if Colab asks.



In [11]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 9.6 MB/s eta 0:00:00


### Learning: Import Libraries

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Next, we import all the required modules from the installed libraries. These include tools for document loading, text splitting, embeddings, vector stores, and various retrieval components.

**Watch for:** If an import fails, re-run the install cell.



In [5]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_402/2860160806.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: Set Up OpenAI API Key

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** To use OpenAI models for embeddings and chat, we need to set up the API key. We will retrieve it from Google Colab's user data secrets for security.

**Watch for:** If an import fails, re-run the install cell.



In [7]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: if not os.environ.get("OPENAI_API_KEY"):

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** This cell ensures that the OpenAI API key is properly configured. If not found in `userdata`, it will prompt the user to enter it.

**Watch for:** Never hardcode secrets in shared notebooks.



In [8]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: Load Research Paper Data

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** We will load a PDF research paper, specifically the Llama 2 paper, for our retrieval demonstrations. This section handles finding the PDF file and defining its path.

**Watch for:** Confirm page count and first-page text look sane.



In [9]:
DATA_DIR = Path(
    r"/content/data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
/content/data/llama2-research-paper.pdf


### Learning: loader = PyPDFLoader(str(PDF_PATH))

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Using `PyPDFLoader`, we load the PDF document. Each page of the PDF is treated as a `Document` object.

**Watch for:** Confirm page count and first-page text look sane.



In [12]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


### Learning: print("First-page metadata:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's inspect the first page's metadata and content to understand the structure of the loaded documents.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [13]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh K

### Learning: Enrich Document Metadata

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** To facilitate more granular filtering and retrieval, we define a helper function to identify the major section of the paper based on its page number. This allows us to add custom metadata to each document.

**Watch for:** Confirm page count and first-page text look sane.



In [14]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

### Learning: PyPDFLoader page index is normally zero-based

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** We iterate through all loaded pages and update their metadata with additional context like the paper name, organization, year, document type, and the identified section.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

### Learning: for page_document in pages[:5]:

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's check the updated metadata for the first few pages to confirm the enrichment.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [16]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/llama2-research-paper.pdf', 'total

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}


### Learning: Chunking Documents

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Large documents need to be split into smaller, manageable chunks for effective retrieval. We use `RecursiveCharacterTextSplitter` to create chunks with specified size and overlap.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


### Learning: for chunk_number, chunk in enumerate(chunks):

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We assign a unique `chunk_id` to each chunk, incorporating its page number and its order within the page, which can be useful for debugging and tracing.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [18]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

### Learning: print("Chunk content:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Let's examine the content and metadata of the first chunk to ensure it's structured as expected.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [19]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

### Learning: Generate Embeddings

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Embeddings convert text into numerical vectors, which are essential for semantic search. We use OpenAI's `text-embedding-3-small` model.

**Watch for:** Never hardcode secrets in shared notebooks.



In [20]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### Learning: test_vector = embeddings.embed_query(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** We'll test the embedding model by embedding a sample query and inspecting the dimensions and a few values of the resulting vector.

**Watch for:** Note dimension size; it must match the index.



In [21]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [0.0028285980224609375, -0.052093505859375, -0.0210418701171875, -0.05535888671875, -0.0264129638671875, 0.028961181640625, -0.0020694732666015625, 0.03472900390625, -0.0164794921875, -0.02447509765625]


### Learning: Create and Persist Vector Store

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** We use Chroma as our vector store to store the document chunks and their embeddings. This allows for efficient similarity search. We also include an option to rebuild the index if needed.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [22]:
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

### Learning: vector_store = Chroma.from_documents(

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** This cell creates the Chroma vector store from our `chunks` and `embeddings`. It will persist the index to disk, so it can be reloaded later without re-embedding.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [23]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: /content/data/chroma_llama2_retriever


### Learning: Load Existing Vector Store

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** If the vector store has already been created and persisted, we can load it directly from the disk. This avoids re-processing the embeddings every time the notebook is run.

**Watch for:** Note dimension size; it must match the index.



In [24]:
# Same directory used during creation
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

### Learning: PERSIST_DIRECTORY

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Verify the persistence directory path.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [25]:
PERSIST_DIRECTORY

PosixPath('/content/data/chroma_llama2_retriever')

### Learning: Load the existing Chroma collection

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** This cell loads the existing Chroma vector store, ensuring we can access the embedded documents.

**Watch for:** Note dimension size; it must match the index.



In [26]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: /content/data/chroma_llama2_retriever


### Learning: Helper Function to Display Documents

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We define a utility function `display_documents` to neatly print the retrieved documents, showing their rank, metadata, and a truncated version of their content. This will help in visualizing retrieval results.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [27]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

### Learning: vector_store.similarity_search()

**What you'll learn:** Set knobs (k, window size, weights) before the experiment.

**What this cell does:** The `vector_store.similarity_search()` method can be used for basic retrieval. We will use it within more advanced retriever configurations.

**Watch for:** Change one knob at a time when you compare runs.



In [ ]:
# vector_store.similarity_search()

### Learning: Similarity Search (Dense Retrieval)

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** We configure a `similarity_retriever` using `vector_store.as_retriever()` with `search_type="similarity"`. This type of retriever fetches documents most similar to the query based on their embedding vectors.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [28]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

### Learning: query = "What model sizes of Llama 2 were released?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Let's perform a similarity search with a specific query and display the top 4 results.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [29]:
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-12
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the open release of LLMs, when done safely, will be a net benefit to society. Like all LLMs,
Llama 2 is

RANK: 2
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-342
SOURCE

### Learning: Maximum Marginal Relevance (MMR) Retrieval

**What you'll learn:** Balance relevance with diversity to reduce near-duplicates.

**What this cell does:** MMR search aims to retrieve documents that are both relevant to the query and diverse among themselves. It balances relevance with diversity to avoid redundancy in results.

**Watch for:** Watch lambda_mult: too high ≈ plain similarity; too low ≈ off-topic.



In [30]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

### Learning: query = "How was Llama 2-Chat trained and aligned?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We'll test the MMR retriever with a query, asking how Llama 2-Chat was trained and aligned. This should return relevant yet diverse documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [31]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 5
SECTION: pretraining
CHUNK ID: llama2-page-5-chunk-14
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdes

RANK: 2
PAPER PAGE: 31
SECTION: safety
CHUNK ID: llama2-page-31-chunk-135
SOURCE: /

### Learning: query = "How was Llama 2-Chat trained and aligned?"

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Let's compare the results from similarity search and MMR for the same query to observe the diversity introduced by MMR.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [32]:
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

### Learning: print("Similarity Search results:")

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This cell prints the paper page, section, and chunk ID for documents returned by both similarity search and MMR, allowing for a direct comparison of their retrieval characteristics.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [33]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
5 pretraining llama2-page-5-chunk-14
3 introduction llama2-page-3-chunk-9
8 fine_tuning llama2-page-8-chunk-29
4 introduction llama2-page-4-chunk-12

MMR results:
5 pretraining llama2-page-5-chunk-14
31 safety llama2-page-31-chunk-135
77 appendix llama2-page-77-chunk-339
34 discussion llama2-page-34-chunk-146


### Learning: Similarity Score Threshold Retrieval

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** This retrieval method only returns documents whose similarity score to the query exceeds a specified threshold. This helps in filtering out less relevant documents.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [34]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64,
    }
)

### Learning: query = "What safety techniques were used for Llama 2-Chat?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We'll query about safety techniques and see which documents meet the `0.64` score threshold.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [35]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual raters. Additionally, these
safety evaluations are performed using content standards that are likely to be biased towards theLlama
2-Chatmodels.
We are releasing the following models to the general publ



### Learning: threshold_retriever = vector_store.as_retriever(

**What you'll learn:** Return only candidates above a relevance floor.

**What this cell does:** This commented out code block shows an example of how you might adjust the `score_threshold` for different results.

**Watch for:** If results are empty, the threshold is too strict for this query.



In [36]:
# threshold_retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 10,
#         "score_threshold": 0.35,
#     }


### Learning: query = "What safety techniques were used for Llama 2-Chat?"

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Instead of using `as_retriever`, we can directly use `similarity_search_with_relevance_scores` on the `vector_store` to get the scores alongside the documents. This provides more control for custom filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [37]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6412
Paper page: 4
Section: introduction
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual ra
Rank: 2
Relevance score: 0.6368
Paper page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3
Rank: 3
Relevance score: 0.6297
Paper page: 2
Section: front_matter
4 Safety 20
4.1 Safe

### Learning: Calculating Embeddings Metrics (Cosine Similarity, Euclidean Distance, Dot Product)

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** This section demonstrates how to manually calculate different similarity metrics between a query embedding and document embeddings, which are the underlying mechanisms for vector search.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [38]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


### Learning: query_vector = np.asarray(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Here, we embed the metric query and the candidate document texts to get their numerical vector representations. We then print their shapes to verify.

**Watch for:** Note dimension size; it must match the index.



In [39]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)


### Learning: def cosine_similarity(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** We define functions to calculate Cosine Similarity, Euclidean Distance, and Dot Product between two vectors. These are common metrics used to quantify the similarity between embeddings.

**Watch for:** Note dimension size; it must match the index.



In [40]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

### Learning: metric_rows = []

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This cell iterates through the candidate documents and calculates all three similarity metrics (cosine similarity, Euclidean distance, and dot product) between the query vector and each document vector. The results are stored in a DataFrame for easy comparison.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [41]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.579142,0.917512,0.579219,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,llama2-page-32-chunk-138,0.538664,0.960562,0.538669,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,llama2-page-17-chunk-73,0.527883,0.971572,0.527725,system message during the conversation by inte...
3,13,fine_tuning,llama2-page-13-chunk-54,0.505156,0.995052,0.505381,evaluating a generative model is an open resea...
4,10,fine_tuning,llama2-page-10-chunk-35,0.500087,0.999762,0.499936,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,llama2-page-10-chunk-40,0.495792,1.004133,0.495728,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_query

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Display the metric query for reference.

**Watch for:** Use the metric your embedding model was trained with.



In [42]:
metric_query

'How was reinforcement learning with human feedback used?'

### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by cosine similarity in descending order to see the most similar documents first. Cosine similarity ranges from -1 (opposite) to 1 (identical).

**Watch for:** Use the metric your embedding model was trained with.



In [43]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.579142,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538664,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527883,system message during the conversation by inte...
3,13,fine_tuning,0.505156,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500087,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495792,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by Euclidean distance in ascending order. Smaller Euclidean distance indicates higher similarity.

**Watch for:** Use the metric your embedding model was trained with.



In [44]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,0.917512,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.960562,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.971572,system message during the conversation by inte...
3,13,fine_tuning,0.995052,evaluating a generative model is an open resea...
4,10,fine_tuning,0.999762,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,1.004133,3.2.2 Reward Modeling The reward model takes a...


### Learning: metric_table.sort_values(

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** Sort the results by dot product in descending order. For normalized vectors, dot product is equivalent to cosine similarity.

**Watch for:** Use the metric your embedding model was trained with.



In [45]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.579219,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538669,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527725,system message during the conversation by inte...
3,13,fine_tuning,0.505381,evaluating a generative model is an open resea...
4,10,fine_tuning,0.499936,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495728,3.2.2 Reward Modeling The reward model takes a...


### Learning: normalized_query_vector = (

**What you'll learn:** See how different similarity metrics score the same pairs.

**What this cell does:** This section explicitly demonstrates that when vectors are normalized, the dot product between them becomes equivalent to their cosine similarity. This is a fundamental concept in vector space models.

**Watch for:** Use the metric your embedding model was trained with.



In [46]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.57914198 0.53866448 0.52788284 0.50515638 0.50008706 0.49579245]

Dot product after normalization:
[0.57914198 0.53866448 0.52788284 0.50515638 0.50008706 0.49579245]

Are they approximately equal? True


### Learning: Filtered Retrieval (Pre-filtering)

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** We can apply filters *before* the similarity search to restrict the search space to documents matching specific metadata criteria (e.g., only documents from the 'fine_tuning' section). This is known as pre-filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [47]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

### Learning: query = "How was Llama 2-Chat aligned with human preferences?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** Let's query how Llama 2-Chat was aligned with human preferences, but only retrieve documents from the `fine_tuning` section.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [48]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 19
SECTION: fine_tuning
CHUNK ID: llama2-page-19-chunk-79
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure12: Humanevaluationresults for Llama 2-Chatmodelscomparedtoopen-andclosed-sourcemodels
across ~4,000 helpfulness prompts with three raters per prompt.
The largestLlama 2-Chat model is competitive with ChatGPT.Llama 2-Chat 70B model has a win rate of
36% and a tie rate of 31.5% relative to ChatGPT.Llama 2-Chat 70B model outperforms PaLM-bison chat
model by a large percentage on our prompt set. More results and analysis is available in Section A.3.7.
Inter-Rater Reliability (IRR). In our human evaluations, three different annotators provided independent
assessments for each model generation comparison. High IRR scores (closer to 1.0) are typically seen as
better from a data quality persp

RANK: 2
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: llama2-page-18-chunk-77
SOU

### Learning: for document in fine_tuning_documents:

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** This assertion verifies that all retrieved documents indeed belong to the 'fine_tuning' section, confirming the filter's effectiveness.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [49]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


### Learning: filtered_retriever = vector_store.as_retriever(

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Here's a more complex pre-filter example using multiple conditions (`$and`): filtering by `section`, `year`, and `organization`.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [50]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

### Learning: query = "How was human preference data collected?"

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We query about human preference data collection, applying the complex pre-filter to retrieve only relevant documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [51]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK: 2
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-38
SOU

Prefilter

## 16. Filtered Retrieval (Post-filtering)

Post-filtering involves retrieving a larger set of candidate documents first, and then applying metadata filters *after* the initial retrieval. This can be useful when the filter conditions are complex or when the vector store doesn't support advanced pre-filtering efficiently.

### Learning: pre_filter = {

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** This cell sets up a pre-filter using the same criteria as before and demonstrates a retriever configured with this pre-filter.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [52]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-42
SOU

Post-Filtering

## 17. Post-filtering Example

First, we retrieve a broader set of candidates without any filters directly applied to the retriever. This retrieves documents from various sections.

### Learning: unfiltered_candidates = vector_store.similarity_search(

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** We fetch 15 candidate documents without any initial filtering.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [53]:
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

### Learning: post_filtered_documents = [

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** After retrieving the broad set of documents, we manually apply the filtering conditions (section and year) in Python. This is post-filtering.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [54]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: /content/data/llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: llama2-page-11-chunk-42
SOU

### Learning: print(

**What you'll learn:** Score broadly, then drop non-matching candidates.

**What this cell does:** This cell compares the number of documents before and after post-filtering, illustrating how post-filtering reduces the set of candidate documents to only those meeting the specified criteria.

**Watch for:** Post-filter can leak restricted docs into the candidate set.



In [55]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 13
